# 📖 Notebook 1: Time-Series Data Modeling

Before we run fancy aggregations, we need to understand **what** time-series data is, **how** it differs from regular relational data, and **why** a specialized database (TimescaleDB) handles it so much better than plain PostgreSQL.

## Learning Objectives

By the end of this notebook you will understand:
- What time-series data looks like (measurements, tags, fields, timestamps)
- How TimescaleDB hypertables automatically partition data by time
- The difference between **tags** (indexed metadata) and **fields** (measured values)
- Why hypertables outperform regular tables for time-series workloads

## 🛠️ Setup

Start the infrastructure first:

```bash
cd deep-dives/time-series-databases
docker-compose up -d
```

### Visualization Tools

- **Adminer** (SQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `timescaledb`, User `demo`, Password `demo`, Database `tsdb_demo`
- **Grafana** (Dashboards): http://localhost:3000  
  Login: `admin` / `admin`. Add a PostgreSQL data source pointing to `timescaledb:5432`.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "tsdb_demo",
    "user": "demo",
    "password": "demo"
}

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

def run_query(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    with get_conn() as conn:
        return pd.read_sql_query(sql, conn, params=params)

def run_exec(sql, params=None):
    """Execute a statement (INSERT / CREATE / etc.)."""
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
        conn.commit()

# Quick connection check
df = run_query("SELECT default_version, installed_version FROM pg_available_extensions WHERE name = 'timescaledb'")
print(f"TimescaleDB version: {df['installed_version'].iloc[0]}")
print("✅ Connected to TimescaleDB!")

---
## 1 — What Is Time-Series Data?

Time-series data is any data where each record has a **timestamp** and represents a measurement at that point in time.  
Think of it like a heart-rate monitor: every second it writes *when* and *what* the reading was.

A time-series data point typically has three parts:

| Part | Purpose | Example |
|------|---------|---------|
| **Timestamp** | When the measurement was taken | `2025-03-30 14:00:00` |
| **Tags** | Metadata you filter/group by (indexed) | `host=server-1`, `region=us-west` |
| **Fields** | The actual measured value | `cpu_usage=45.2` |

> **Rule of thumb**: Tags = *what produced the data*. Fields = *what was measured*.

Let's look at the data the `init.sql` script created for us.

In [ ]:
# Peek at the raw data
df = run_query("""
    SELECT time, host, region, metric_name, value
    FROM metrics
    ORDER BY time DESC
    LIMIT 10
""")
df

In [ ]:
# How much data do we have?
stats = run_query("""
    SELECT
        count(*)                        AS total_rows,
        count(DISTINCT host)            AS hosts,
        count(DISTINCT metric_name)     AS metrics,
        min(time)                       AS earliest,
        max(time)                       AS latest
    FROM metrics
""")
print(f"Total rows     : {stats['total_rows'].iloc[0]:,}")
print(f"Distinct hosts : {stats['hosts'].iloc[0]}")
print(f"Distinct metrics: {stats['metrics'].iloc[0]}")
print(f"Time range     : {stats['earliest'].iloc[0]} → {stats['latest'].iloc[0]}")

---
## 2 — Hypertables: Automatic Time Partitioning

A **hypertable** looks and feels like a normal PostgreSQL table, but under the hood TimescaleDB automatically splits it into **chunks** — one chunk per time interval.

This is exactly the *time-based partitioning* concept from the article:
- **Writes** always go to the latest chunk → fast sequential appends.
- **Reads** skip chunks outside the query's time range → less data to scan.
- **Retention** = dropping old chunks instead of running expensive `DELETE` queries.

Let's see the chunks TimescaleDB created for our `metrics` hypertable.

In [ ]:
chunks = run_query("""
    SELECT
        chunk_name,
        range_start,
        range_end,
        pg_size_pretty(total_bytes) AS size
    FROM timescaledb_information.chunks
    WHERE hypertable_name = 'metrics'
    ORDER BY range_start
""")
print(f"TimescaleDB created {len(chunks)} chunks for 7 days of data:\n")
chunks

Each chunk is a separate physical table on disk.  
When you query `WHERE time > now() - interval '1 hour'`, TimescaleDB only touches the **latest chunk** and ignores all the older ones.

---
## 3 — Tags vs. Fields and the Cardinality Trap

In our schema:
- **Tags** (`host`, `region`) are columns you filter or group by. TimescaleDB indexes them.
- **Fields** (`value`) are the measured numbers. They are *not* indexed.

### Why does this matter?

Every unique combination of tags creates a **series**.  
10 hosts × 2 regions × 4 metrics = 80 series — very manageable.

But if you added a `request_id` tag with millions of unique values, the number of series explodes — this is called **cardinality explosion**. It eats memory and kills query performance.

> **Golden rule**: Only use tags for **low-cardinality** metadata (host, region, service).  
> High-cardinality identifiers (user IDs, request IDs) should be fields, not tags.

In [ ]:
# Count the distinct series in our dataset
series = run_query("""
    SELECT host, region, metric_name, count(*) AS points
    FROM metrics
    GROUP BY host, region, metric_name
    ORDER BY host, metric_name
""")
print(f"Total distinct series: {len(series)}")
series.head(12)

---
## 4 — Inserting Time-Series Data

Inserts into a hypertable look exactly like normal SQL `INSERT` statements.  
TimescaleDB transparently routes each row to the correct chunk based on its timestamp.

In [ ]:
# Insert a single data point
run_exec("""
    INSERT INTO metrics (time, host, region, metric_name, value)
    VALUES (now(), 'server-99', 'eu-west', 'cpu_usage', 72.5)
""")

# Verify it landed
run_query("SELECT * FROM metrics WHERE host = 'server-99' ORDER BY time DESC LIMIT 1")

In [ ]:
# Batch insert — the typical pattern for high-throughput ingestion
import random, datetime

now = datetime.datetime.now(datetime.timezone.utc)
rows = []
for i in range(1000):
    ts = now - datetime.timedelta(seconds=i * 30)
    rows.append((ts, 'server-99', 'eu-west', 'cpu_usage', 40 + random.gauss(0, 10)))

with get_conn() as conn:
    with conn.cursor() as cur:
        cur.executemany(
            "INSERT INTO metrics (time, host, region, metric_name, value) VALUES (%s,%s,%s,%s,%s)",
            rows
        )
    conn.commit()

count = run_query("SELECT count(*) AS n FROM metrics WHERE host = 'server-99'")
print(f"server-99 now has {count['n'].iloc[0]:,} data points")

---
## 5 — Hypertable vs. Plain Table Performance

The `init.sql` script created a second table called `metrics_plain` — a regular PostgreSQL table with the exact same data.  
Let's run the same query on both and compare.

In [ ]:
QUERY = """
    SELECT
        date_trunc('hour', time) AS hour,
        avg(value) AS avg_cpu
    FROM {table}
    WHERE host = 'server-1'
      AND metric_name = 'cpu_usage'
      AND time > now() - interval '24 hours'
    GROUP BY hour
    ORDER BY hour
"""

for table in ['metrics', 'metrics_plain']:
    sql = QUERY.format(table=table)
    start = time.perf_counter()
    df = run_query(sql)
    elapsed = (time.perf_counter() - start) * 1000
    print(f"{table:20s} → {elapsed:6.1f} ms  ({len(df)} rows)")

In [ ]:
# Let's look at the query plan to see WHY the hypertable is faster
plan = run_query(f"EXPLAIN (ANALYZE, FORMAT TEXT) {QUERY.format(table='metrics')}")
print("=== Hypertable (metrics) ===")
for row in plan.iloc[:, 0]:
    print(row)

print("\n")
plan2 = run_query(f"EXPLAIN (ANALYZE, FORMAT TEXT) {QUERY.format(table='metrics_plain')}")
print("=== Plain table (metrics_plain) ===")
for row in plan2.iloc[:, 0]:
    print(row)

### What to look for in the plans

- The **hypertable** plan shows **chunk exclusion**: only 1-2 chunks are scanned.
- The **plain table** plan scans the entire table even though we only need the last 24 hours.

This is the power of time-based partitioning — the database *skips* data that can't possibly match your time filter.

---
## 6 — Visualize a Time Series

Let's plot CPU usage for one host over the last 24 hours to see what time-series data looks like.

In [ ]:
cpu = run_query("""
    SELECT time, value
    FROM metrics
    WHERE host = 'server-1'
      AND metric_name = 'cpu_usage'
      AND time > now() - interval '24 hours'
    ORDER BY time
""")

plt.figure(figsize=(14, 4))
plt.plot(cpu['time'], cpu['value'], linewidth=0.5, alpha=0.8)
plt.title('CPU Usage — server-1 (last 24 h)')
plt.xlabel('Time')
plt.ylabel('CPU %')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🧠 Key Takeaways

1. **Time-series data** = timestamp + tags (metadata) + fields (measured values).
2. **Hypertables** look like normal tables but are automatically partitioned into time-based chunks.
3. Chunk exclusion lets queries skip irrelevant time ranges → faster reads.
4. Keep tags **low-cardinality** to avoid the cardinality explosion problem.
5. Inserts work with normal SQL — TimescaleDB routes them to the right chunk.

**Next notebook →** We'll use `time_bucket()` and window functions to run powerful aggregations over this data.